### OPEX Postprocessing

In [1]:
#!/usr/bin/env python3
"""
Create histograms of total project OPEX and annualised OPEX
for each macro scenario (Scenario.name) and combined across scenarios.

Definitions:
- "Macro scenario" = Scenario.name (e.g. "Best-practice O&M", "Harsh env + immature O&M", ...)
- Each macro scenario has multiple replicates; each replicate is a scenario entry
  with its own scenario_id and parquet file.

Assumptions:
- scenarios.json exists in the RESULTS_FOLDER and contains a list of scenarios
  with at least:
      - "scenario_id"
      - "overrides" dict that includes "Scenario.name" (preferred)
- For each scenario_id SID there is a parquet file named:
      {TABLE_NAME}_df_{SID}.parquet
- Each parquet file contains OPEX records with at least:
      - "timestamp" (datetime-like)
      - "OM_payment" (numeric, usually negative for outflow)

For each macro scenario S:
- We compute per replicate:
    - total project OPEX (sum of OM_payment, converted to positive EUR)
    - annualised OPEX (total / years over which timestamps span)

Outputs (in RESULTS_FOLDER):
- Per macro scenario:
    - histogram_total_opex_<ScenarioNameSanitized>.png
    - histogram_annual_opex_<ScenarioNameSanitized>.png
- Combined (all macro scenarios overlaid, colored with legend):
    - histogram_total_opex_all_scenarios.png
    - histogram_annual_opex_all_scenarios.png
"""

from pathlib import Path
import json
from typing import Dict, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------------------
# 1) Configuration
# --------------------------------------------------------------------
RESULTS_FOLDER = r"M:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\results\OPEX_Uncertainty"
TABLE_NAME = "opex"  # prefix of parquet files: opex_df_<scenario_id>.parquet

RESULTS_FOLDER = Path(RESULTS_FOLDER)
assert RESULTS_FOLDER.exists(), f"Results folder not found: {RESULTS_FOLDER}"

SCENARIOS_PATH = RESULTS_FOLDER / "scenarios.json"
assert SCENARIOS_PATH.exists(), f"scenarios.json not found at: {SCENARIOS_PATH}"

# --------------------------------------------------------------------
# 2) Load scenarios.json
# --------------------------------------------------------------------
with open(SCENARIOS_PATH, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

# Might be {"scenarios": [...]} or just [...]
if isinstance(scenarios, dict) and "scenarios" in scenarios:
    scenarios = scenarios["scenarios"]

assert isinstance(scenarios, list), "Expected a list of scenarios in scenarios.json"
print(f"Loaded {len(scenarios)} scenario entries from {SCENARIOS_PATH}")

# --------------------------------------------------------------------
# 3) Helpers
# --------------------------------------------------------------------
def get_macro_name(sc: Dict[str, Any], default: str) -> str:
    """
    Extract the macro scenario name from a scenario entry.

    Priority:
    1. sc["overrides"]["Scenario.name"]  (if present)
    2. sc.get("Scenario.name")
    3. sc.get("label")
    4. fallback: default (e.g. scenario_id)
    """
    overrides = sc.get("overrides") or {}
    name = overrides.get("Scenario.name") or sc.get("Scenario.name") \
           or sc.get("label") or default
    return str(name)


def _sanitize_name(name: str) -> str:
    """Make a string safe for filenames."""
    return "".join(c if c.isalnum() or c in "-_" else "_" for c in name)


def _maybe_millions_single(ax, values: np.ndarray, label: str) -> np.ndarray:
    """Scale a single array to millions if needed, and set x-axis label."""
    if values.size == 0:
        ax.set_xlabel(label)
        return values
    vmax = float(np.max(values))
    if vmax >= 1e6:
        ax.set_xlabel(label + " [million]")
        return values / 1e6
    else:
        ax.set_xlabel(label)
        return values


def _maybe_millions_multi(ax, arrays: List[np.ndarray], label: str) -> List[np.ndarray]:
    """Scale multiple arrays jointly based on global max."""
    nonempty = [arr for arr in arrays if arr.size > 0]
    if not nonempty:
        ax.set_xlabel(label)
        return arrays
    all_vals = np.concatenate(nonempty)
    vmax = float(np.max(all_vals))
    if vmax >= 1e6:
        ax.set_xlabel(label + " [million]")
        return [arr / 1e6 for arr in arrays]
    else:
        ax.set_xlabel(label)
        return arrays


def _total_and_annual_opex(df: pd.DataFrame) -> tuple[float, float]:
    """
    Given an OPEX_records DataFrame with columns:
        - 'timestamp'
        - 'OM_payment' (usually negative for cash outflow)

    Returns:
        (total_opex_eur_positive, annualised_opex_eur_per_year_positive)
    """
    if "OM_payment" not in df.columns:
        raise ValueError("Expected 'OM_payment' column in OPEX parquet.")

    df = df.copy()
    df["OM_payment"] = pd.to_numeric(df["OM_payment"], errors="coerce").fillna(0.0)

    total = float(df["OM_payment"].sum())

    # Interpret OM_payment as outflow; ensure positive magnitude
    total_opex = -total if total <= 0.0 else total

    # Annualise based on timestamp span (if timestamps exist)
    if "timestamp" in df.columns:
        ts = pd.to_datetime(df["timestamp"], errors="coerce")
        ts = ts.dropna()
        if len(ts) >= 2:
            dt_years = (ts.max() - ts.min()).days / 365.25
            years = dt_years if dt_years > 0 else 1.0
        else:
            years = 1.0
    else:
        years = 1.0

    annual_opex = total_opex / years
    return total_opex, annual_opex

# --------------------------------------------------------------------
# 4) Group scenario_ids by macro scenario name
# --------------------------------------------------------------------
macro_to_entries: Dict[str, Dict[str, Any]] = {}  # macro_name -> sid -> scenario_entry

for sc in scenarios:
    sid = str(sc.get("scenario_id", "")).strip()
    if not sid:
        continue
    macro_name = get_macro_name(sc, default=sid)
    # deduplicate by scenario_id within each macro scenario
    macro_to_entries.setdefault(macro_name, {})
    macro_to_entries[macro_name][sid] = sc

print("\nMacro scenarios found (unique scenario_ids):")
for macro, entries_by_sid in macro_to_entries.items():
    print(f"  {macro}: {len(entries_by_sid)} replicate(s)")

if not macro_to_entries:
    raise RuntimeError("No macro scenarios could be identified; check Scenario.name in scenarios.json")

# --------------------------------------------------------------------
# 5) Load parquet data and aggregate per macro scenario
# --------------------------------------------------------------------
macro_data: Dict[str, Dict[str, Any]] = {}
missing_files: List[Any] = []

for macro_name, entries_by_sid in macro_to_entries.items():
    total_opex_list: List[float] = []    # one value per replicate
    annual_opex_list: List[float] = []   # one value per replicate

    for sid, sc in entries_by_sid.items():
        parquet_path = RESULTS_FOLDER / f"{TABLE_NAME}_df_{sid}.parquet"
        if not parquet_path.exists():
            missing_files.append((macro_name, sid, str(parquet_path), "Missing file"))
            continue

        try:
            df = pd.read_parquet(parquet_path)
        except Exception as e:
            missing_files.append((macro_name, sid, str(parquet_path), f"Read error: {e}"))
            continue

        try:
            total_opex, annual_opex = _total_and_annual_opex(df)
        except Exception as e:
            print(
                f"WARNING: macro '{macro_name}', scenario_id {sid} could not compute OPEX: {e}"
            )
            continue

        total_opex_list.append(total_opex)
        annual_opex_list.append(annual_opex)

    total_opex_arr = np.array(total_opex_list, dtype=float)
    annual_opex_arr = np.array(annual_opex_list, dtype=float)

    if total_opex_arr.size or annual_opex_arr.size:
        macro_data[macro_name] = {
            "total_opex": total_opex_arr,
            "annual_opex": annual_opex_arr,
        }

        print(
            f"\nMacro scenario '{macro_name}': "
            f"{total_opex_arr.size} replicate total-OPEX values, "
            f"{annual_opex_arr.size} annualised OPEX values"
        )

print(f"\nCollected OPEX data for {len(macro_data)} macro scenario(s).")
if missing_files:
    print("\nFiles not loaded:")
    for m in missing_files:
        print(f" - macro='{m[0]}', scenario_id={m[1]}, path={m[2]}, reason={m[3]}")

if not macro_data:
    raise RuntimeError("No usable macro scenario OPEX data; cannot plot.")

# --------------------------------------------------------------------
# 6) Per-macro-scenario histograms (with Gaussian curve fit)
# --------------------------------------------------------------------
print("\nCreating per-macro-scenario OPEX histograms...")

for macro_name, data in macro_data.items():
    total_opex_arr = data["total_opex"]
    annual_opex_arr = data["annual_opex"]

    safe_name = _sanitize_name(macro_name)

    # 6a) Histogram of total project OPEX across replicates
    if total_opex_arr.size > 0:
        plt.figure(figsize=(8, 5))
        ax = plt.gca()
        vals = _maybe_millions_single(ax, total_opex_arr, "Total project OPEX [EUR]")

        bins = min(30, max(1, vals.size))

        # histogram
        n, bin_edges, _ = ax.hist(
            vals,
            bins=bins,
            edgecolor="C0",
            alpha=0.6,
            linewidth=1.2,
            label="Histogram",
        )

        # Gaussian fit
        if vals.size > 1:
            mu = float(vals.mean())
            sigma = float(vals.std(ddof=1))
            if sigma > 0:
                x = np.linspace(bin_edges[0], bin_edges[-1], 200)
                bin_width = bin_edges[1] - bin_edges[0]
                y = (
                    1.0
                    / (sigma * np.sqrt(2.0 * np.pi))
                    * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
                    * vals.size
                    * bin_width
                )
                ax.plot(x, y, color="C0", linewidth=1.8, label="Gaussian fit")

        ax.set_ylabel("Frequency")
        ax.set_title(f"Total project OPEX\nScenario: {macro_name}")
        ax.grid(True, linestyle="--", alpha=0.3)
        ax.legend()
        out_path = RESULTS_FOLDER / f"histogram_total_opex_{safe_name}.png"
        plt.tight_layout()
        plt.savefig(out_path, dpi=150)
        plt.close()
        print(f"  Saved total OPEX histogram for '{macro_name}' to: {out_path}")

    # 6b) Histogram of annualised OPEX across replicates
    if annual_opex_arr.size > 0:
        plt.figure(figsize=(8, 5))
        ax = plt.gca()
        vals = _maybe_millions_single(ax, annual_opex_arr, "Annualised OPEX [EUR/year]")

        bins = min(30, max(1, vals.size))

        # histogram
        n, bin_edges, _ = ax.hist(
            vals,
            bins=bins,
            edgecolor="C1",
            alpha=0.6,
            linewidth=1.2,
            label="Histogram",
        )

        # Gaussian fit
        if vals.size > 1:
            mu = float(vals.mean())
            sigma = float(vals.std(ddof=1))
            if sigma > 0:
                x = np.linspace(bin_edges[0], bin_edges[-1], 200)
                bin_width = bin_edges[1] - bin_edges[0]
                y = (
                    1.0
                    / (sigma * np.sqrt(2.0 * np.pi))
                    * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
                    * vals.size
                    * bin_width
                )
                ax.plot(x, y, color="C1", linewidth=1.8, label="Gaussian fit")

        ax.set_ylabel("Frequency")
        ax.set_title(f"Annualised OPEX\nScenario: {macro_name}")
        ax.grid(True, linestyle="--", alpha=0.3)
        ax.legend()
        out_path = RESULTS_FOLDER / f"histogram_annual_opex_{safe_name}.png"
        plt.tight_layout()
        plt.savefig(out_path, dpi=150)
        plt.close()
        print(f"  Saved annual OPEX histogram for '{macro_name}' to: {out_path}")

# --------------------------------------------------------------------
# 7) Combined histograms (all macro scenarios overlaid, with fits)
# --------------------------------------------------------------------
print("\nCreating combined OPEX histograms for all macro scenarios...")

macro_names = list(macro_data.keys())
total_arrays = [macro_data[m]["total_opex"] for m in macro_names]
annual_arrays = [macro_data[m]["annual_opex"] for m in macro_names]

# use matplotlib default color cycle
color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# 7a) Combined total project OPEX histogram
nonempty_total = [a for a in total_arrays if a.size > 0]
if nonempty_total:
    plt.figure(figsize=(9, 6))
    ax = plt.gca()
    scaled_total_arrays = _maybe_millions_multi(ax, total_arrays, "Total project OPEX [EUR]")

    nonempty_scaled = [a for a in scaled_total_arrays if a.size > 0]
    all_vals = np.concatenate(nonempty_scaled)
    bins = min(30, max(5, len(all_vals)))
    bin_edges = np.linspace(all_vals.min(), all_vals.max(), bins)

    for i, (macro_name, arr) in enumerate(zip(macro_names, scaled_total_arrays)):
        if arr.size == 0:
            continue

        color = color_cycle[i % len(color_cycle)]

        # histogram (step)
        n, _, _ = ax.hist(
            arr,
            bins=bin_edges,
            histtype="step",
            linewidth=1.5,
            alpha=0.9,
            label=macro_name,
            color=color,
        )

        # Gaussian fit
        if arr.size > 1:
            mu = float(arr.mean())
            sigma = float(arr.std(ddof=1))
            if sigma > 0:
                x = np.linspace(bin_edges[0], bin_edges[-1], 200)
                bin_width = bin_edges[1] - bin_edges[0]
                y = (
                    1.0
                    / (sigma * np.sqrt(2.0 * np.pi))
                    * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
                    * arr.size
                    * bin_width
                )
                ax.plot(x, y, color=color, linestyle="-", linewidth=1.5)

    ax.set_ylabel("Frequency")
    ax.set_title("Total project OPEX\n(all macro scenarios)")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend()
    out_path = RESULTS_FOLDER / "histogram_total_opex_all_scenarios.png"
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"Saved combined total OPEX histogram to: {out_path}")
else:
    print("No total project OPEX data available; skipping combined total OPEX plot.")

# 7b) Combined annualised OPEX histogram
nonempty_ann = [a for a in annual_arrays if a.size > 0]
if nonempty_ann:
    plt.figure(figsize=(9, 6))
    ax = plt.gca()
    scaled_ann_arrays = _maybe_millions_multi(ax, annual_arrays, "Annualised OPEX [EUR/year]")

    nonempty_scaled_ann = [a for a in scaled_ann_arrays if a.size > 0]
    all_vals_ann = np.concatenate(nonempty_scaled_ann)
    bins_ann = min(30, max(5, len(all_vals_ann)))
    bin_edges_ann = np.linspace(all_vals_ann.min(), all_vals_ann.max(), bins_ann)

    for i, (macro_name, arr) in enumerate(zip(macro_names, scaled_ann_arrays)):
        if arr.size == 0:
            continue

        color = color_cycle[i % len(color_cycle)]

        # histogram (step)
        n, _, _ = ax.hist(
            arr,
            bins=bin_edges_ann,
            histtype="step",
            linewidth=1.5,
            alpha=0.9,
            label=macro_name,
            color=color,
        )

        # Gaussian fit
        if arr.size > 1:
            mu = float(arr.mean())
            sigma = float(arr.std(ddof=1))
            if sigma > 0:
                x = np.linspace(bin_edges_ann[0], bin_edges_ann[-1], 200)
                bin_width = bin_edges_ann[1] - bin_edges_ann[0]
                y = (
                    1.0
                    / (sigma * np.sqrt(2.0 * np.pi))
                    * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
                    * arr.size
                    * bin_width
                )
                ax.plot(x, y, color=color, linestyle="-", linewidth=1.5)

    ax.set_ylabel("Frequency")
    ax.set_title("Annualised OPEX\n(all macro scenarios)")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend()
    out_path = RESULTS_FOLDER / "histogram_annual_opex_all_scenarios.png"
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"Saved combined annual OPEX histogram to: {out_path}")
else:
    print("No annualised OPEX data available; skipping combined annual OPEX plot.")

print("\nDone.")


AssertionError: Results folder not found: M:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\results\OPEX_Uncertainty